In [31]:
from helpers import *
import pandas as pd
csv_df = pd.read_csv('customers_sales.csv')

display(csv_df)


,customer_name,customer_branch,file_name,order_quantity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800
5,PrecisionCut GmbH,Stuttgart Branch,Geo (13).DXF,7650
6,AutoParts AG,Berlin Branch,Geo (14).DXF,7250
7,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
8,MetalForm SA,Frankfurt Branch,Geo (17).DXF,1900
9,MetalForm SA,Frankfurt Branch,Geo (19).DXF,1450


In [32]:
customer_branch_id = {}


#print(customer_branch_id)
#csv_df['customer_id'] = csv_df['customer_name'].map(customer_id)
#csv_df['branch_id'] = csv_df['customer_branch'].map(branch_id)

display(csv_df)

,customer_name,customer_branch,file_name,order_quantity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800
5,PrecisionCut GmbH,Stuttgart Branch,Geo (13).DXF,7650
6,AutoParts AG,Berlin Branch,Geo (14).DXF,7250
7,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
8,MetalForm SA,Frankfurt Branch,Geo (17).DXF,1900
9,MetalForm SA,Frankfurt Branch,Geo (19).DXF,1450


In [36]:
csv_df = pd.read_csv('customers_sales.csv')

folder = "10991360/"



cutting_speed_mm_min = 1000 # mm/min
offset_general_s = 120 #s 
offset_time_smartgate_s = 2 #s
offset_time_sortmaster_s = 5 #s
offset_time_manually_s = 20 #s
offset_time_hole_s = 3 #s

smartgate_min = (30, 30) # min
smartgate_max = (160, 160) # max

sortmaster_min = (60, 90) # min
sortmaster_max = (1500, 2000) # max

def get_cutting_time(path):
    cutting_time_s = get_cutting_length(path) / (cutting_speed_mm_min / 60)

    extraction_method = get_extraction_method(path)

    if extraction_method == "Gate + Master":
        offset_extraction_method_s = min(offset_time_smartgate_s, offset_time_sortmaster_s)
    elif extraction_method == "Master":
        offset_extraction_method_s = offset_time_sortmaster_s
    elif extraction_method == "Gate":
        offset_extraction_method_s = offset_time_smartgate_s
    else:
        offset_extraction_method_s = offset_time_manually_s

    offset_time_total_s = offset_general_s + offset_extraction_method_s + get_number_of_holes(path) * offset_time_hole_s

    return cutting_time_s + offset_time_total_s

def get_fit(object, min_opening, max_opening):
    ob_1, ob_2 = sorted(object)
    min_1, min_2 = sorted(min_opening)
    max_1, max_2 = sorted(max_opening)

    return min_1 <= ob_1 <= max_1 and min_2 <= ob_2 <= max_2

def get_extraction_method(path):

    bounding_box = tuple(sorted(get_dxf_bound(path)))

    if get_fit(bounding_box, sortmaster_min, sortmaster_max) and get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate + Master"
    
    elif get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate"
    
    elif get_fit(bounding_box, sortmaster_min, sortmaster_max):
        return "Master"
    
    else:
        return "None"

def get_bounding_box_area(path):
    bounding_box = get_dxf_bound(path)
    return bounding_box[0] * bounding_box[1]

def get_automation_index(extraction_method):

    if extraction_method != "None":
        return 1
    else:
        return 0

csv_df['path'] = folder + csv_df['file_name']

csv_df[['bounding_box_width_mm', 'bounding_box_height_mm']] = csv_df['path'].apply(lambda p: pd.Series(get_dxf_bound(p)))

csv_df['bounding_box_area_mm^2'] = csv_df['bounding_box_width_mm'] * csv_df['bounding_box_height_mm']
csv_df['part_area_mm^2'] = csv_df['path'].apply(get_part_area)
csv_df['rectangularity_score'] = csv_df['bounding_box_height_mm'] / csv_df['bounding_box_width_mm']
csv_df['filling_score'] = csv_df['part_area_mm^2'] / csv_df['bounding_box_area_mm^2']

csv_df['extraction_method'] = csv_df["path"].apply(get_extraction_method)
csv_df['automation_index'] = csv_df["extraction_method"].map(get_automation_index)

csv_df['number_of_holes'] = csv_df["path"].apply(get_number_of_holes)
csv_df['cutting_length_mm'] = csv_df["path"].apply(get_cutting_length)
csv_df['cutting_time_s'] = csv_df["path"].apply(get_cutting_time)

In [37]:
pd.set_option('display.max_rows', None) 
display(csv_df)


,customer_name,customer_branch,file_name,order_quantity,path,bounding_box_width_mm,bounding_box_height_mm,bounding_box_area_mm^2,part_area_mm^2,rectangularity_score,filling_score,extraction_method,automation_index,number_of_holes,cutting_length_mm,cutting_time_s
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,10991360/Geo (1).DXF,1600.219048,305.139765,4.882905e+05,4.013592e+04,0.190686,0.082197,Master,1,0,3438.024977,331.281499
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,10991360/Geo (10).DXF,853.830303,333.401400,2.846682e+05,2.061794e+05,0.390477,0.724280,Master,1,0,3576.198184,339.571891
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,10991360/Geo (11).DXF,164.000000,145.120247,2.379972e+04,1.428161e+04,0.884880,0.600075,Master,1,0,1965.579365,242.934762
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,10991360/Geo (15).DXF,193.800891,138.442589,2.683030e+04,1.840244e+04,0.714355,0.685883,Master,1,0,1302.774283,203.166457
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,10991360/Geo (12).DXF,937.150697,74.006665,6.935540e+04,6.005769e+04,0.078970,0.865941,Master,1,9,2815.159795,320.909588
5,PrecisionCut GmbH,Stuttgart Branch,Geo (13).DXF,7650,10991360/Geo (13).DXF,166.136100,67.260858,1.117446e+04,5.875069e+03,0.404854,0.525759,Master,1,2,790.858441,178.451506
6,AutoParts AG,Berlin Branch,Geo (14).DXF,7250,10991360/Geo (14).DXF,1591.022804,88.903140,1.414469e+05,1.135475e+05,0.055878,0.802757,Master,1,0,5658.389545,464.503373
7,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,10991360/Geo (15).DXF,193.800891,138.442589,2.683030e+04,1.840244e+04,0.714355,0.685883,Master,1,0,1302.774283,203.166457
8,MetalForm SA,Frankfurt Branch,Geo (17).DXF,1900,10991360/Geo (17).DXF,1179.078462,305.093782,3.597295e+05,2.003054e+05,0.258756,0.556822,Master,1,11,5304.131255,476.247875
9,MetalForm SA,Frankfurt Branch,Geo (19).DXF,1450,10991360/Geo (19).DXF,1152.727829,314.994358,3.631028e+05,3.631028e+05,0.273260,1.000000,Master,1,0,2935.444373,301.126662


In [29]:
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
#csv_df = csv_df.sort_values(by="cutting_length_mm", ascending=False)
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
display(csv_df)


,customer_name,customer_branch,file_name,order_quantity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800
...,...,...,...,...
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100
